# 09장 오디오증폭기

* 이름: 홍길동
* 학번: 2025123456
* 조: 1

## 공통 코드

In [ ]:
import sys
from pathlib import Path

# common_dir = Path.cwd().parent / "ece2knu_common"
common_dir = Path.cwd()
sys.path.insert(0, str(common_dir))

from common import *

*******************************************************************************

## 9.4 예비 문제

### 9.4.1 CC증폭기 회로에 대하여 민감도 해석을 하시오.

* 경우1: R_1 = 6.8 kOhm => R_1 = 6 kOhm
* 경우2: R_1 = 6.8 kOhm => R_1 = 8 kOhm

In [ ]:
display_image("./images/ch09_04_01_circuit.png", width=700)

* 교재 그림 9.6 직류해석. CC증폭기는 R_C = 0 이다 (컬렉터가 V_CC 에 직접 연결된다).

In [ ]:
V_CC_V = 15
R_2_Ohm = 10000
R_E_Ohm = 390
beta = 210
V_BE_V = 0.65


def analyze_cc(R_1_Ohm):
    """CC증폭기 직류 해석 (교재 9.3절 순서). 값을 출력하고 dict 로 돌려준다."""
    # 1) 테브난 등가회로
    V_BB_V = R_2_Ohm / (R_1_Ohm + R_2_Ohm) * V_CC_V
    R_B_Ohm = (R_1_Ohm * R_2_Ohm) / (R_1_Ohm + R_2_Ohm)
    # 2) 입력측 KVL
    I_B_A = (V_BB_V - V_BE_V) / (R_B_Ohm + (1 + beta) * R_E_Ohm)
    # 3) 트랜지스터의 특성
    I_C_A = beta * I_B_A
    # 4) 출력측 KVL (I_E = (1 + beta) * I_B)
    I_E_A = (1 + beta) * I_B_A
    V_CE_V = V_CC_V - R_E_Ohm * I_E_A

    print_value("V_BB_V", V_BB_V)
    print_value("R_B_kOhm", R_B_Ohm / 1000)
    print_value("I_B_uA", I_B_A * 1000000)
    print_value("I_C_mA", I_C_A * 1000)
    print_value("V_CE_V", V_CE_V)
    return {"R_1_Ohm": R_1_Ohm, "V_BB_V": V_BB_V, "R_B_Ohm": R_B_Ohm, "I_B_A": I_B_A, "I_C_A": I_C_A, "V_CE_V": V_CE_V}


base = analyze_cc(6800)

In [ ]:
case_1 = analyze_cc(6000)

In [ ]:
case_2 = analyze_cc(8000)

In [ ]:
# 민감도: 기본 (R_1 = 6.8 kOhm) 에 대한 변화율 [%]
def change_percent(new_value, old_value):
    return (new_value - old_value) / old_value * 100


for name, case in (("경우1", case_1), ("경우2", case_2)):
    print(name)
    print_value("  R_1 변화율 [%]", change_percent(case["R_1_Ohm"], base["R_1_Ohm"]), decimals=1, width=7)
    print_value("  I_C 변화율 [%]", change_percent(case["I_C_A"], base["I_C_A"]), decimals=1, width=7)
    print_value("  V_CE 변화율 [%]", change_percent(case["V_CE_V"], base["V_CE_V"]), decimals=1, width=7)

In [ ]:
display_image("./data/ch09_04_01_case_1.png", width=600)

In [ ]:
display_image("./data/ch09_04_01_case_2.png", width=600)

|                  | 해석            | 해석 (경우1)    | 해석 (경우2)    |
|------------------|-----------------|-----------------|-----------------|
| V_BB (V)         | 8.93            | ??              | ??              |
| R_B (kOhm)       | 4               | ??              | ??              |
| I_B (uA)         | 95.9            | ??              | ??              |
| I_C (mA)         | 20.13           | ??              | ??              |
| V_CE (V)         | 7.15            | ??              | ??              |
| I_C 변화율 (%)   |                 | ??              | ??              |
| V_CE 변화율 (%)  |                 | ??              | ??              |

* R_1 변화율: 경우1 -11.8 %, 경우2 +17.6 %
* V_CE 는 I_E = (1 + beta) I_B 로 계산한 값이다. 교재 표의 V_CE = 7.15 V 는 I_E ≈ I_C 로 근사해 계산한 값이다 (15 - 0.39 x 20.13 = 7.15).
* 결론: R_1 이 -11.8 % / +17.6 % 변해도 I_C 는 +5.8 % / -7.6 % 만 변한다.